# DDR analysis under lock v5.1 (runs once)

Checks the lock, the inference record, all 33 prediction hashes and the split hash, then computes exactly what lock v5.1 specifies:

- **P1 (primary):** matched-volume net PDR misses avoided at f = 1.31 x referable prevalence (prevalence from half A), all DDR images, 110 ordered pairs x 9 seed combinations, max-percentile-rank ensemble, group-cluster bootstrap (2,000).
- **P2 (co-primary, descriptive):** union rule at f: anchor-relative PDR rescued, incremental referrals and incremental false positives.
- **Secondary, intraset selection:** C, rho_cost and partner marginal cost on half A; anchor-relative PDR rescued under the union rule on half B; within-anchor Spearman, architecture bootstrap, partial, permutation null; rule comparison including the two-stage rule; swap sensitivity.
- **Secondary, transport:** P1 and P2 at fixed 25%; EyePACS estimation-pool C against the DDR half-B outcome.
- **Also reported:** grade 3 or worse in place of PDR; without retfound_adversarial.

Bootstrap intervals for P1/P2 hold each model's referral set fixed at the full-sample threshold (conditional on thresholds), resampling filename groups. Refuses to run twice. CPU is enough; about 10 to 15 minutes.

In [1]:
# Cell 1: guards
from google.colab import drive
drive.mount('/content/drive')
import os, json, hashlib, itertools, datetime
import numpy as np, pandas as pd
from scipy.stats import rankdata

BASE = '/content/drive/MyDrive/stochastic_orthogonality/data/dr_ensemble_v2'
SPLITS, PREDS, RESULTS, AUDIT = (f'{BASE}/{d}' for d in ('01_splits','03_predictions','04_results','99_audit'))
LOCK_FILE = f'{RESULTS}/locked_config_v5_1_ddr.json'
REC_FILE  = f'{AUDIT}/ddr_inference_record_v2.json'
SPLIT_FILE = f'{SPLITS}/ddr_split_intraset_v2.csv'
EST_FILE  = f'{RESULTS}/pair_quantities_estimation_v4.csv'
OUT_FILE  = f'{RESULTS}/ddr_results_v5_1.json'
def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for blk in iter(lambda: f.read(chunk), b''): h.update(blk)
    return h.hexdigest()
assert not os.path.exists(OUT_FILE), 'DDR analysis already run once. Do not rerun.'
REC = json.load(open(REC_FILE)); LOCK_SHA = sha256_file(LOCK_FILE)
assert LOCK_SHA == REC['lock_v5_1_sha256'], 'Lock changed since inference.'
assert sha256_file(SPLIT_FILE) == REC['split_file_sha256'], 'Split changed since inference.'
for fn, h in REC['prediction_sha256'].items():
    assert sha256_file(f'{PREDS}/{fn}') == h, f'{fn} changed since inference.'
ROSTER = ['densenet121_binary','densenet121_5class','efficientnet_b3_binary','efficientnet_b3_5class',
          'resnet50_binary','resnet50_5class','vit_base_binary','vit_base_5class',
          'retfound_binary','retfound_5class','retfound_adversarial']
SEEDS = ['', '_seed1', '_seed2']; W = {4: 500, 3: 200, 2: 15}
ORDERED = [(a, b) for a in ROSTER for b in ROSTER if a != b]
N_BOOT_CLUSTER, N_BOOT_ARCH, N_PERM, RNG = 2000, 1000, 10000, 20260924
print('All hashes verified against the inference record. Lock v5.1', LOCK_SHA[:16])

Mounted at /content/drive
All hashes verified against the inference record. Lock v5.1 2531e1336f9487ee


In [2]:
# Cell 2: load split and predictions (the one read)
sp = pd.read_csv(SPLIT_FILE, dtype={'image_id': str}).sort_values('image_id').reset_index(drop=True)
IDS = sp.image_id.to_numpy(); grade = sp.grade.to_numpy().astype(int); half = sp.half.to_numpy()
grp_codes, grp_idx = np.unique(sp.group.to_numpy(), return_inverse=True)
n = len(IDS); referable = grade >= 2; pdr = grade == 4; sev3 = grade >= 3
WMISS = np.zeros(n); [WMISS.__setitem__(grade == g, w) for g, w in W.items()]
P = {}
for a in ROSTER:
    for s in SEEDS:
        df = pd.read_csv(f'{PREDS}/ddr_predictions_{a}{s}_v2.csv', dtype={'image_id': str}).set_index('image_id').reindex(IDS)
        assert df.predicted_probability.notna().all(); P[f'{a}{s}'] = df.predicted_probability.to_numpy(np.float64)
A, B = half == 'A', half == 'B'
PREV_A = float(referable[A].mean()); F = 1.31 * PREV_A
OPENED = datetime.datetime.now(datetime.timezone.utc).isoformat()
print(f'DDR: {n:,} images, {pdr.sum()} PDR, {len(grp_codes)} groups | half A {A.sum():,} (PDR {pdr[A].sum()}), half B {B.sum():,} (PDR {pdr[B].sum()})')
print(f'Referable prevalence on A {PREV_A:.4f} -> operating point f = {F:.4f}. Opened {OPENED}')

DDR: 12,422 images, 913 PDR, 332 groups | half A 6,205 (PDR 382), half B 6,217 (PDR 531)
Referable prevalence on A 0.4237 -> operating point f = 0.5550. Opened 2026-09-23T15:58:34.254795+00:00


In [3]:
# Cell 3: shared machinery
def refer(score, frac, mask=None):
    idx = np.arange(n) if mask is None else np.where(mask)[0]
    k = int(round(frac * len(idx))); m = np.zeros(n, bool); m[idx[np.argsort(-score[idx], kind='stable')[:k]]] = True; return m
def pct(score, mask=None):
    out = np.zeros(n); idx = np.arange(n) if mask is None else np.where(mask)[0]; out[idx] = rankdata(score[idx]) / len(idx); return out
def combos(a, b): return [(f'{a}{sa}', f'{b}{sb}') for sa in SEEDS for sb in SEEDS]

def benefit_vectors(frac, target, mask=None):
    '''Per-image contributions averaged over 110 ordered pairs x 9 seeds.'''
    ref = {k: refer(v, frac, mask) for k, v in P.items()}; rk = {k: pct(v, mask) for k, v in P.items()}
    keep = np.ones(n, bool) if mask is None else mask
    D = {k: np.zeros(n) for k in ('mv_net','anchor_miss','ens_miss','rescued','incr_ref','incr_fp','mv_net_vs_better')}
    cnt = 0; better = []
    for a, b in ORDERED:
        for ka, kb in combos(a, b):
            ra, rb = ref[ka], ref[kb]; e = refer(np.maximum(rk[ka], rk[kb]), frac, mask)
            am = target & ~ra & keep; em = target & ~e & keep
            D['anchor_miss'] += am; D['ens_miss'] += em; D['mv_net'] += am.astype(float) - em
            D['rescued'] += target & ~ra & rb & keep; D['incr_ref'] += rb & ~ra & keep; D['incr_fp'] += rb & ~ra & ~referable & keep
            bm = (target & ~rb & keep)
            D['mv_net_vs_better'] += (am if am.sum() <= bm.sum() else bm).astype(float) - em
            cnt += 1
    return {k: v / cnt for k, v in D.items()}

def cluster_summary(D, mask=None):
    keep = np.ones(n, bool) if mask is None else mask
    tot = {k: float(v[keep].sum()) for k, v in D.items()}
    G = {k: np.bincount(grp_idx, weights=v * keep, minlength=len(grp_codes)) for k, v in D.items()}
    rng = np.random.default_rng(RNG); bs = {k: [] for k in D}
    for _ in range(N_BOOT_CLUSTER):
        w = np.bincount(rng.integers(0, len(grp_codes), len(grp_codes)), minlength=len(grp_codes))
        for k in D: bs[k].append(float((G[k] * w).sum()))
    out = {k: {'per_pair_mean': round(tot[k], 3), 'cluster_95': [round(float(np.percentile(bs[k], 2.5)), 3), round(float(np.percentile(bs[k], 97.5)), 3)]} for k in D}
    out['relative_reduction_vs_anchor'] = round(tot['mv_net'] / tot['anchor_miss'], 4) if tot['anchor_miss'] else None
    out['union_rescue_fraction_of_anchor_misses'] = round(tot['rescued'] / tot['anchor_miss'], 4) if tot['anchor_miss'] else None
    out['incr_fp_per_pdr_rescued'] = round(tot['incr_fp'] / tot['rescued'], 1) if tot['rescued'] else None
    out['incr_referrals_per_pdr_rescued'] = round(tot['incr_ref'] / tot['rescued'], 1) if tot['rescued'] else None
    return out
print('ready')

ready


In [4]:
# Cell 4: P1 and P2 (primary), then transport T1 and grade>=3
RES = {}
for label, frac in (('prevalence_relative_f', F), ('fixed_25', 0.25)):
    for tlabel, target in (('pdr', pdr), ('grade3plus', sev3)):
        D = benefit_vectors(frac, target); S = cluster_summary(D)
        RES[f'{label}__{tlabel}'] = {'fraction_referred_each': round(frac, 4), **S}
P1 = RES['prevalence_relative_f__pdr']['mv_net']
P1_SUPPORTED = bool(P1['per_pair_mean'] > 0 and P1['cluster_95'][0] > 0)
for k, v in RES.items():
    print(f"{k:34s} f={v['fraction_referred_each']:.3f}  matched-volume net {v['mv_net']['per_pair_mean']:+.2f} {v['mv_net']['cluster_95']}"
          f"  | relative {v['relative_reduction_vs_anchor']}  | union rescued {v['rescued']['per_pair_mean']:.2f}, incr FP {v['incr_fp']['per_pair_mean']:.0f}")
print('\nP1 SUPPORTED:', P1_SUPPORTED)

prevalence_relative_f__pdr         f=0.555  matched-volume net +12.41 [7.193, 18.251]  | relative 0.2739  | union rescued 25.37, incr FP 790
prevalence_relative_f__grade3plus  f=0.555  matched-volume net +13.32 [8.02, 19.15]  | relative 0.2807  | union rescued 27.03, incr FP 790
fixed_25__pdr                      f=0.250  matched-volume net +13.36 [6.111, 21.922]  | relative 0.0496  | union rescued 90.52, incr FP 97
fixed_25__grade3plus               f=0.250  matched-volume net +17.38 [9.879, 26.586]  | relative 0.0589  | union rescued 102.71, incr FP 97

P1 SUPPORTED: True


In [5]:
# Cell 5: intraset selection (predictors on one half, outcome on the other)
def predictors(mask, frac):
    ref = {k: refer(v, frac, mask) for k, v in P.items()}
    marg = {a: float(np.mean([WMISS[referable & ~ref[f'{a}{s}'] & mask].sum() for s in SEEDS])) for a in ROSTER}
    Q = {}
    for a, b in itertools.combinations(ROSTER, 2):
        Cs, rs = [], []
        for ka, kb in combos(a, b):
            ra, rb = ref[ka], ref[kb]
            mA = WMISS[referable & ~ra & mask].sum(); mB = WMISS[referable & ~rb & mask].sum(); mb = WMISS[referable & ~ra & ~rb & mask].sum()
            Cs.append(1 - mb / min(mA, mB) if min(mA, mB) > 0 else np.nan)
            ca = np.where(mask & referable & ~ra, WMISS, 0) + np.where(mask & ~referable & ra, 1, 0)
            cb = np.where(mask & referable & ~rb, WMISS, 0) + np.where(mask & ~referable & rb, 1, 0)
            rs.append(np.corrcoef(ca[mask], cb[mask])[0, 1])
        Q[(a, b)] = Q[(b, a)] = {'C': float(np.nanmean(Cs)), 'rho_cost': float(np.nanmean(rs))}
    return Q, marg
def outcomes(mask, frac):
    ref = {k: refer(v, frac, mask) for k, v in P.items()}; rk = {k: pct(v, mask) for k, v in P.items()}; O = {}
    for a, b in ORDERED:
        acc = {'rescued': [], 'incr_fp': [], 'mv_net': []}
        for ka, kb in combos(a, b):
            ra, rb = ref[ka], ref[kb]; e = refer(np.maximum(rk[ka], rk[kb]), frac, mask)
            acc['rescued'].append((pdr & ~ra & rb & mask).sum()); acc['incr_fp'].append((rb & ~ra & ~referable & mask).sum())
            acc['mv_net'].append((pdr & ~ra & mask).sum() - (pdr & ~e & mask).sum())
        O[(a, b)] = {k: float(np.mean(v)) for k, v in acc.items()}
    return O
def spear(x, y, z=None):
    x, y = rankdata(x), rankdata(y)
    if z is not None:
        Z = np.column_stack([np.ones(len(z)), rankdata(z)])
        x = x - Z @ np.linalg.lstsq(Z, x, rcond=None)[0]; y = y - Z @ np.linalg.lstsq(Z, y, rcond=None)[0]
    return float(np.corrcoef(x, y)[0, 1]) if np.std(x) > 0 and np.std(y) > 0 else np.nan
def within(ms, Q, marg, O, pred, partial=False, roster=ROSTER):
    vals = []
    for a in ms:
        cands = [c for c in ms if c != a and c in roster]
        if len(set(cands)) < 3: continue
        x = [(-marg[c] if pred == 'quality' else Q[(a, c)][pred] * (-1 if pred == 'rho_cost' else 1)) for c in cands]
        y = [O[(a, c)]['rescued'] for c in cands]; z = [-marg[c] for c in cands] if partial else None
        r = spear(x, y, z); vals.append(r) if np.isfinite(r) else None
    return float(np.mean(vals)) if vals else np.nan
def arch_boot(fn, roster):
    rng = np.random.default_rng(RNG); out = [fn(list(rng.choice(roster, len(roster), replace=True))) for _ in range(N_BOOT_ARCH)]
    return [round(float(np.nanpercentile(out, 2.5)), 4), round(float(np.nanpercentile(out, 97.5)), 4)]
def perm_null(Q, O, roster):
    rng = np.random.default_rng(RNG + 1); obs = []
    for _ in range(N_PERM):
        v = [spear(rng.permutation([Q[(a, c)]['C'] for c in roster if c != a]), [O[(a, c)]['rescued'] for c in roster if c != a]) for a in roster]
        obs.append(np.nanmean(v))
    return [round(float(np.percentile(obs, 2.5)), 4), round(float(np.percentile(obs, 97.5)), 4)]
def rules(Q, marg, O, roster):
    per = {}
    for a in roster:
        cands = [c for c in roster if c != a]; mmin = min(marg[c] for c in cands)
        tier = [c for c in cands if marg[c] <= 1.10 * mmin]
        pick = {'least_ceiling_overlap': max(cands, key=lambda c: Q[(a, c)]['C']),
                'least_weighted_dependence': min(cands, key=lambda c: Q[(a, c)]['rho_cost']),
                'best_partner_marginal': min(cands, key=lambda c: marg[c]),
                'two_stage': max(tier, key=lambda c: (Q[(a, c)]['C'], -marg[c]))}
        per[a] = {r: O[(a, c)] for r, c in pick.items()}
        per[a]['random'] = {k: float(np.mean([O[(a, c)][k] for c in cands])) for k in ('rescued','incr_fp','mv_net')}
        per[a]['oracle_rescued'] = float(max(O[(a, c)]['rescued'] for c in cands)); per[a]['picks'] = pick
    def dboot(r1, r2, k):
        d = np.array([per[a][r1][k] - per[a][r2][k] for a in roster]); rng = np.random.default_rng(RNG + 2)
        bs = [rng.choice(d, len(d), replace=True).mean() for _ in range(N_BOOT_ARCH)]
        return {'mean': round(float(d.mean()), 3), 'arch_95': [round(float(np.percentile(bs, 2.5)), 3), round(float(np.percentile(bs, 97.5)), 3)],
                'anchors_favouring_first': int((d > 0).sum())}
    rn = ['least_ceiling_overlap','least_weighted_dependence','best_partner_marginal','two_stage','random']
    return {'rule_mean_rescued': {r: round(float(np.mean([per[a][r]['rescued'] for a in roster])), 3) for r in rn} |
                                 {'oracle': round(float(np.mean([per[a]['oracle_rescued'] for a in roster])), 3)},
            **{f'{r}_minus_best_marginal__{k}': dboot(r, 'best_partner_marginal', k)
               for r in ('least_ceiling_overlap', 'two_stage') for k in ('rescued', 'mv_net', 'incr_fp')},
            'picks': {a: per[a]['picks'] for a in roster}}
def selection(Q, marg, O, roster):
    res = {}
    for lab, pred, partial in (('C', 'C', False), ('C_partial_on_partner_marginal', 'C', True),
                               ('rho_cost_negated', 'rho_cost', False), ('partner_quality', 'quality', False)):
        est = within(list(roster), Q, marg, O, pred, partial, roster)
        res[lab] = {'estimate': round(est, 4), 'arch_bootstrap_95': arch_boot(lambda ms, p=pred, q=partial: within(ms, Q, marg, O, p, q, roster), roster)}
    res['C']['permutation_null_95'] = perm_null(Q, O, roster)
    res['C']['supported'] = bool(res['C']['estimate'] > 0 and res['C']['arch_bootstrap_95'][0] > 0)
    res['rules'] = rules(Q, marg, O, roster); return res

QA, MA = predictors(A, F); OB = outcomes(B, F)
NOADV = [a for a in ROSTER if a != 'retfound_adversarial']
SEL = {'primary_A_to_B': selection(QA, MA, OB, ROSTER), 'without_retfound_adversarial': selection(QA, MA, OB, NOADV)}
QB, MB = predictors(B, F); OA = outcomes(A, F)
SEL['swap_B_to_A'] = selection(QB, MB, OA, ROSTER)
print(json.dumps({k: {kk: vv for kk, vv in v.items() if kk != 'rules'} for k, v in SEL.items()}, indent=1))
print(json.dumps({k: v for k, v in SEL['primary_A_to_B']['rules'].items() if k != 'picks'}, indent=1))

{
 "primary_A_to_B": {
  "C": {
   "estimate": 0.3602,
   "arch_bootstrap_95": [
    -0.0897,
    0.7032
   ],
   "permutation_null_95": [
    -0.1978,
    0.195
   ],
   "supported": false
  },
  "C_partial_on_partner_marginal": {
   "estimate": 0.44,
   "arch_bootstrap_95": [
    -0.0711,
    0.7174
   ]
  },
  "rho_cost_negated": {
   "estimate": 0.4477,
   "arch_bootstrap_95": [
    -0.0418,
    0.7863
   ]
  },
  "partner_quality": {
   "estimate": 0.4868,
   "arch_bootstrap_95": [
    -0.0357,
    0.704
   ]
  }
 },
 "without_retfound_adversarial": {
  "C": {
   "estimate": 0.2225,
   "arch_bootstrap_95": [
    -0.2927,
    0.6159
   ],
   "permutation_null_95": [
    -0.218,
    0.2155
   ],
   "supported": false
  },
  "C_partial_on_partner_marginal": {
   "estimate": 0.3096,
   "arch_bootstrap_95": [
    -0.2776,
    0.6451
   ]
  },
  "rho_cost_negated": {
   "estimate": 0.3717,
   "arch_bootstrap_95": [
    -0.207,
    0.7873
   ]
  },
  "partner_quality": {
   "estimate": 0

In [6]:
# Cell 6: transport T2 (EyePACS estimation-pool predictors against the DDR half-B outcome)
est = pd.read_csv(EST_FILE); QE, ME = {}, {}
for _, r in est.iterrows():
    QE[(r.model_a, r.model_b)] = QE[(r.model_b, r.model_a)] = {'C': float(r.C), 'rho_cost': float(r.rho_cost)}
    ME[r.model_a] = float(r.marginal_a); ME[r.model_b] = float(r.marginal_b)
SEL['transport_eyepacs_predictors_to_DDR_B'] = selection(QE, ME, OB, ROSTER)
print(json.dumps({k: v for k, v in SEL['transport_eyepacs_predictors_to_DDR_B'].items() if k != 'rules'}, indent=1))

{
 "C": {
  "estimate": 0.2929,
  "arch_bootstrap_95": [
   -0.1782,
   0.7266
  ],
  "permutation_null_95": [
   -0.1967,
   0.1935
  ],
  "supported": false
 },
 "C_partial_on_partner_marginal": {
  "estimate": 0.4146,
  "arch_bootstrap_95": [
   -0.2058,
   0.8052
  ]
 },
 "rho_cost_negated": {
  "estimate": 0.3458,
  "arch_bootstrap_95": [
   -0.1717,
   0.7791
  ]
 },
 "partner_quality": {
  "estimate": 0.463,
  "arch_bootstrap_95": [
   -0.0349,
   0.728
  ]
 }
}


In [7]:
# Cell 7: write results once, log
out = {'lock_v5_1_sha256': LOCK_SHA, 'opened_utc': OPENED, 'cohort': {'images': int(n), 'pdr': int(pdr.sum()), 'groups': int(len(grp_codes)),
       'half_A_images': int(A.sum()), 'half_B_images': int(B.sum()), 'referable_prevalence_A': round(PREV_A, 4), 'operating_point_f': round(F, 4)},
       'fellow_eye_check': REC['fellow_eye_check'],
       'P1_supported': P1_SUPPORTED, 'benefit': RES, 'selection': SEL,
       'pre_registered_prediction': LOCK['operating_points']['pre_registered_prediction'] if (LOCK := json.load(open(LOCK_FILE))) else None,
       'notes': ['per_pair_mean is the mean over 110 ordered pairs x 9 seed combinations of per-pair counts over the evaluated images',
                 'cluster intervals resample the 332 filename groups with referral sets fixed at full-sample thresholds',
                 'mv_net_vs_better compares the ensemble with the better single reader of each combination (v3-style comparator)']}
with open(OUT_FILE, 'w') as f: json.dump(out, f, indent=2)
with open(f'{AUDIT}/RUN_LOG.md', 'a') as f:
    f.write(f'''\n\n## DDR analysis, lock v5.1 ({OPENED[:10]})\n\nRun once. P1 (matched-volume net PDR at f={F:.3f}): {P1['per_pair_mean']} {P1['cluster_95']}; '''
            f'''supported = {P1_SUPPORTED}. Intraset selection C: {SEL['primary_A_to_B']['C']['estimate']} {SEL['primary_A_to_B']['C']['arch_bootstrap_95']}. '''
            f'''Output 04_results/ddr_results_v5_1.json. Produced by 99_audit/DDR_analysis_v5_1.ipynb.\n''')
print('Wrote', OUT_FILE)

Wrote /content/drive/MyDrive/stochastic_orthogonality/data/dr_ensemble_v2/04_results/ddr_results_v5_1.json


## After this notebook
Send back `04_results/ddr_results_v5_1.json`. Move this executed notebook to `99_audit/DDR_analysis_v5_1.ipynb`. It refuses to run a second time.